# Becke Partition 二阶梯度简单理解

In [1]:
from pyscf import gto, dft, lib, grad, hessian, data
import numpy as np
from functools import partial

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()

In [3]:
def get_grids(xyz):
    mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()
    grids = dft.grid.Grids(mol)
    grids.radii_adjust = dft.radi.becke_atomic_radii_adjust
    grids.build(sort_grids=False)
    return mol, grids

## PySCF 数值二阶梯度导数

目前使用的是 PySCF 中用于计算 VV10 导数所用到的函数。但需要留意，由于 PySCF 目前没有实现二阶解析格点导数，我们有必要自己实现一版。为此，我们需要比较完善的函数，计算所有二阶梯度；而不是像一阶梯度那样只需要稍微验证一下。

In [4]:
xyz_orig = np.array([[0.0, 0.0, 0.0], [1.0, 0.1, 0.2], [0.3, 1.1, 0.2], [0.1, 0.1, 1.2]])  # in angstrom

def perturb_xyz(xyz, A, t, sgn, delta):
    xyz_pert = xyz.copy()
    xyz_pert[A, t] += sgn * delta
    return xyz_pert

def perturb_mol(A, t, sgn, delta):
    xyz_coords = perturb_xyz(xyz_orig, A, t, sgn, delta)
    atm_symbols = ["N", "H", "H", "H"]
    xyz_str = "\n".join(f"{atm} {x:.6f} {y:.6f} {z:.6f}" for atm, (x, y, z) in zip(atm_symbols, xyz_coords))
    mol_pert = gto.Mole(atom=xyz_str, basis="def2-TZVP", max_memory=32000).build()
    return mol_pert

def perturb_mol_grids(A, t, sgn, delta):
    mol_pert = perturb_mol(A, t, sgn, delta)
    grids_pert = dft.grid.Grids(mol_pert)
    grids_pert.radii_adjust = dft.radi.becke_atomic_radii_adjust
    grids_pert.build(sort_grids=False)
    return mol_pert, grids_pert

首先我们拿到没有坐标微扰的分子与格点。

In [5]:
mol, grids = perturb_mol_grids(A=0, t=0, sgn=1, delta=0.0)

In [6]:
nonpad_mask = grids.atm_idx != -1
quadrature_weights = grids.quadrature_weights[nonpad_mask]
grid_coords = grids.coords[nonpad_mask]
atm_coords = mol.atom_coords()
atm_indices = grids.atm_idx[nonpad_mask]

natm = atm_coords.shape[0]
ngrids = grid_coords.shape[0]
becke_radii_adjust = dft.radi.becke_atomic_radii_adjust(mol, grids.atomic_radii)
radii_table = np.array([becke_radii_adjust(i, j, 0) for i in range(natm) for j in range(natm)]).reshape(natm, natm)

微扰的分子与格点放在一个字典中。

In [7]:
interval = 3e-5
dict_perturb = {(A, t, sgn): perturb_mol_grids(A, t, sgn, delta=interval) for A in range(len(xyz_orig)) for t in range(3) for sgn in [-1, 1]}
dict_w = {(A, t, sgn): grids_pert.weights for (A, t, sgn), (_, grids_pert) in dict_perturb.items()}
dict_dw = {(A, t, sgn): hessian.rks.get_dweight_dA(mol_pert, grids_pert) for (A, t, sgn), (mol_pert, grids_pert) in dict_perturb.items()}

我们可以先验证一下一阶梯度量的数值导数是否正确。

In [8]:
dw_ndiff = np.zeros((natm, 3, ngrids))
for A in range(natm):
    for t in range(3):
        dw_plus = dict_w[(A, t, 1)][..., nonpad_mask]
        dw_minus = dict_w[(A, t, -1)][..., nonpad_mask]
        dw_ndiff[A, t] = (dw_plus - dw_minus) / (2 * interval / data.nist.BOHR)
assert np.allclose(dw_ndiff, hessian.rks.get_dweight_dA(mol, grids)[..., nonpad_mask])

二阶梯度量可以通过一阶梯度量的数值导数来计算。

In [9]:
ddw_ndiff = np.zeros((natm, 3, natm, 3, ngrids))
for A in range(natm):
    for t in range(3):
        dw_plus = dict_dw[(A, t, 1)][..., nonpad_mask]
        dw_minus = dict_dw[(A, t, -1)][..., nonpad_mask]
        ddw_ndiff[A, t] = (dw_plus - dw_minus) / (2 * interval / data.nist.BOHR)

我们可以通过检查对称性来简单地从一个角度验证正确性，以及估算数值误差大小。下面的误差估计是比较保守的。

In [10]:
assert np.allclose(ddw_ndiff, ddw_ndiff.transpose(2, 3, 0, 1, 4), rtol=1e-4, atol=5e-7)  # Check symmetry

最后我们对称化该数值梯度，作为参考值。

- `ddw_ref` 二阶格点权重梯度，维度 $(A, t, B, s, g)$ `(natm, 3, natm, 3, ngrids)`。

    $$
    \frac{\partial^2 w_g}{\partial R_{At} \partial R_{Bs}}
    $$

In [11]:
ddw_ref = 0.5 * (ddw_ndiff + ddw_ndiff.transpose(2, 3, 0, 1, 4))

## Becke Partition 二阶梯度实现与公式对应

我们需要再来一次。

### 1. 梯度无关量

- `wquad` $w_g^\text{quad}$：原始 Lebedev 权重，维度 $(g,)$ `(ngrids,)`

- `a` $a_{AB}$：Becke radii 矫正表，维度 $(A, B)$ `(natm, natm)`

In [12]:
wquad = quadrature_weights
a = radii_table

### 2. 原子间距离 $\Vert R \Vert_{AB}$

- `atm_coords` $R_{A t}$：原子坐标，维度 $(A, 3)$ `(natm, 3)`

- `atom_dist` $\Vert R \Vert_{AB}$：原子间距离，维度 $(A, B)$ `(natm, natm)`；其只作为分母出现，对角元设为 $\infty$，避免除零

    $$
    \Vert R \Vert_{AB} = 
    \begin{cases}
    \sqrt{\sum_t (R_{B t} - R_{A t})^2} & A \neq B \\
    \infty & A = B
    \end{cases}
    $$

In [13]:
atom_dist = np.linalg.norm(atm_coords[:, None, :] - atm_coords[None, :, :], axis=-1)
for i in range(natm):
    atom_dist[i, i] = np.inf

- $\Vert \partial R \Vert_{A B t}$ `dR_atom_dist` 原子距离导数，导数只对原子 $A$ 进行，维度 $(A, B, t)$ `(natm, natm, 3)`：

    $$
    \Vert \partial R \Vert_{A B t} := \frac{\partial \Vert R \Vert_{AB}}{\partial R_{A t}} = \frac{R_{A t} - R_{B t}}{\Vert R \Vert_{AB}}
    $$

In [14]:
dR_atom_dist = (atm_coords[:, None, :] - atm_coords[None, :, :]) / atom_dist[:, :, None]
assert np.allclose(- dR_atom_dist.swapaxes(0, 1), dR_atom_dist)

在处理原子距离的二阶导数前，我们重新回顾一下一阶导数。我们之所以要用 $\Vert \partial R \Vert_{A B t}$ 这种不二不三的形式，是因为严格来说 $B = A$ 的导数情况我们没有考虑。但对于当前的原子距离问题，$B = A$ 的情况是没有意义的 (同原子没有贡献)。
$$
\begin{align*}
\frac{\partial \Vert R \Vert_{M N}}{\partial R_{A t}} &= \delta_{A M} \frac{\partial \Vert R \Vert_{M N}}{\partial R_{M t}} + \delta_{A N} \frac{\partial \Vert R \Vert_{M N}}{\partial R_{N t}} \quad (M \text{ unrelated to } N) \\
&= \delta_{A M} \Vert \partial R \Vert_{A N t} - \delta_{A N} \Vert \partial R \Vert_{A M t} \\
\end{align*}
$$